In [1]:
import pandas as pd
import numpy as np
import sys
from pathlib import Path
PROJECT_ROOT = Path().resolve().parents[0]
sys.path.append(str(PROJECT_ROOT))

from src.simulation.order_stream import OrderStream
from src.simulation.simulator import Simulator
from src.dynamicProgramming.dp_scheduler import DPScheduler
from src.dynamicProgramming.insertion_policy import GreedyInsertionPolicy
from src.dynamicProgramming.value_function import ValueFunction
from src.dynamicProgramming.feature_extractor import FeatureExtractor
from src.routing.route_utils import route_duration

data_path = Path("../data/features/olist/delivery_jobs_dataset.csv")

In [2]:
# loading a sample dataset
jobs = pd.read_csv(
    data_path,
    parse_dates = [
        'ready_time',
        'due_date'
    ]
)

# dropping rows without lat & lng coordinates
jobs = jobs.dropna(
    subset = [
        'pickup_lat',
        'pickup_lng',
        'delivery_lat',
        'delivery_lng'
    ]
)

# filtering a sample dataset
jobs = jobs.sort_values('ready_time').head(500).copy()
start = pd.Timestamp("2026-01-01 08:00:00")

jobs['ready_time'] = start + pd.to_timedelta(
    np.arange(len(jobs)) * 5,
    unit = 'm'
)

jobs['due_date'] = jobs['ready_time'] + pd.Timedelta(hours = 4)
jobs['service_time_min'] = 5
display(jobs.head())

,job_id,order_id,seller_id,customer_id,pickup_lat,pickup_lng,delivery_lat,delivery_lng,ready_time,due_date,service_time_min,demand
30505,bfbd0f9bdef84302105ad712db648a6c,bfbd0f9bdef84302105ad712db648a6c,ecccfa2bb93b34a3bf033cc5d1dcdc69,86dc2ffce2dfff336de2f386a786e574,-25.507014,-49.275963,-20.585751,-47.863693,2026-01-01 08:00:00,2026-01-01 12:00:00,5,3.0
63623,1ff217aa612f6cd7c4255c9bfe931c8b,1ff217aa612f6cd7c4255c9bfe931c8b,4b1eaadf791bdbbad8c4a35b65236d52,b3a9bf200375f53cc5c6991919c356fd,-21.177710,-47.767820,-23.719311,-46.660397,2026-01-01 08:05:00,2026-01-01 12:05:00,5,1.0
6706,cd3b8574c82b42fc8129f6d502690c3e,cd3b8574c82b42fc8129f6d502690c3e,b499c00f28f4b7069ff6550af8c1348a,7812fcebfc5e8065d31e1bb5f0017dae,-22.600004,-47.407129,-23.032142,-45.570461,2026-01-01 08:10:00,2026-01-01 12:10:00,5,1.0
66586,ed8c7b1b3eb256c70ce0c74231e1da88,ed8c7b1b3eb256c70ce0c74231e1da88,5b179e9e8cc7ab6fd113a46ca584da81,da0ba2a9935bca5b4610b0e3bca9d3b4,-23.568771,-46.698110,-23.453962,-46.731884,2026-01-01 08:15:00,2026-01-01 12:15:00,5,1.0
87892,d207cc272675637bfed0062edffd0818,d207cc272675637bfed0062edffd0818,cca3071e3e9bb7d12640c9fbe2301306,b8cf418e97ae795672d326288dfab7a7,-21.757321,-48.829744,-22.892792,-47.173849,2026-01-01 08:20:00,2026-01-01 12:20:00,5,1.0


In [3]:
# initializing simulation components
stream = OrderStream(jobs)
policy = GreedyInsertionPolicy()

scheduler = DPScheduler(
    insertion_policy = policy,
    value_function = None,
    gamma = 0.95,
    n_couriers = 5
)

simulator = Simulator(stream, scheduler)
feature_extractor = FeatureExtractor()

In [4]:
# cost function
def compute_total_cost(state):
    total_cost = 0

    for courier in state.couriers:
        total_cost += route_duration(
            route = courier.route,
            start_time = courier.current_time
        )

        total_cost += len(courier.completed_jobs) * 5
    
    return total_cost

In [5]:
# preparing predictors and target variables for training
X, y = [], []

state = simulator.initialize(
    start_time = jobs['ready_time'].min()
)

end_time = state.current_time + pd.Timedelta(hours = 6)

while state.current_time < end_time:

    # extracting features
    features = feature_extractor.extract(state)

    # simulate one step
    simulator.step(state, step_minutes=5)

    # computing cost after simulation step
    total_cost = compute_total_cost(state)

    # safety check
    features = np.array(features, dtype = float)
    if np.any(np.isnan(features)) or np.any(np.isinf(features)):
        continue

    # skipping bad samples
    if np.isnan(total_cost) or np.isinf(total_cost):
        continue

    # restricting extreme costs
    total_cost = min(total_cost, 1000)

    # collecting data after simulation
    X.append(features)
    y.append(total_cost)


In [6]:
# training XGBoost Model
X = np.array(X)
y = np.array(y)

# shuffle dataset
idx = np.random.permutation(len(X))
X = X[idx]
y = y[idx]

# train/test split
split = int(len(X) * 0.8)

X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

# normalizing target variable
y = y / 100.0

vf = ValueFunction()
vf.fit(X_train, y_train)

print('XGB training completed!')

for i in range(5):
    pred = vf.predict(X_train[i])
    print(f"Pred: {pred:.2f}, Actual: {y_train[i]:.2f}")

XGB training completed!
Pred: 45.33, Actual: 45.00
Pred: 36.85, Actual: 35.00
Pred: 139.47, Actual: 130.00
Pred: 576.91, Actual: 595.00
Pred: 44.97, Actual: 45.00
